# Support Copilot: GPU model server (Colab)



## 1. Install

In [ ]:
!pip install -q "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

## 2. Download the model

In [ ]:
import os
from huggingface_hub import hf_hub_download

GGUF_REPO = "hossam3759180/support-copilot-gguf"
GGUF_FILENAME = "support-copilot-q4.gguf"

model_path = hf_hub_download(GGUF_REPO, GGUF_FILENAME)
print(model_path)

support-copilot-q4.gguf: reconstructing file:   0%|          |  0.00B / 4.92GB            

support-copilot-q4.gguf: downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/models--hossam3759180--support-copilot-gguf/snapshots/355ab374d84cf3e05459914c98007b959a8792f1/support-copilot-q4.gguf


## 3. Start the model server

In [ ]:
import glob, subprocess, time, requests
!pkill -f llama_cpp.server
!pkill -f cloudflared
time.sleep(2)

libs = glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/*/lib")
env = dict(os.environ, LD_LIBRARY_PATH=":".join(libs) + ":" + os.environ.get("LD_LIBRARY_PATH", ""))

server = subprocess.Popen(
    ["python", "-m", "llama_cpp.server",
     "--model", model_path,
     "--n_gpu_layers", "99",
     "--n_ctx", "1024",
     "--port", "8001"],
    env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
ready = False
for _ in range(90):
    if server.poll() is not None:
        break
    try:
        if requests.get("http://localhost:8001/v1/models", timeout=2).ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if ready:
    print("model server is up")
else:
    print("model server FAILED to start. Last log lines:\n")
    print(open("server.log").read()[-2000:])

model server is up


## 4. Open the tunnel and print the `.env` lines

In [ ]:
import re

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8001"],
                          stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

public_url = None
for _ in range(45):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        public_url = m.group(0)
        break

if public_url:
    print("Put these two lines in your local .env, then restart the app:\n")
    print("LLM_BACKEND=vllm")
    print(f"VLLM_BASE_URL={public_url}/v1")
    print(f"\nCheck in your browser (wait ~30 s if you see Bad gateway): {public_url}/v1/models")
else:
    print("Tunnel did not start. Log:\n")
    print(open("tunnel.log").read()[-1500:])

Put these two lines in your local .env, then restart the app:

LLM_BACKEND=vllm
VLLM_BASE_URL=https://participants-granny-little-lewis.trycloudflare.com/v1

Check in your browser (wait ~30 s if you see Bad gateway): https://participants-granny-little-lewis.trycloudflare.com/v1/models


## 5. Quick test (runs inside Colab, against the local port)

In [ ]:
t0 = time.time()
r = requests.post("http://localhost:8001/v1/completions", json={
    "model": "x",
    "prompt": "<|system|>\nYou are a helpful customer support assistant.\n<|user|>\nQuery: where is my refund?\n<|assistant|>\n",
    "max_tokens": 60,
    "temperature": 0.0,
    "stop": ["<|user|>", "<|system|>"],
})
print(r.json()["choices"][0]["text"].strip())
print(f"\n{time.time() - t0:.1f} s")

I'm on it! I understand that you're looking for information about your refund. To assist you with this, could you please provide me with some additional details? Specifically, can you let me know the order number or any other relevant information related to your refund request? This will help me locate the

2.3 s


## 6. Keep alive
It checks every minute that the server and tunnel are still up, and stops with a message if either dies. Interrupt it (stop button) when you are done.

In [ ]:

while True:
    if server.poll() is not None:
        print("model server stopped."); break
    if tunnel.poll() is not None:
        print("tunnel stopped. ."); break
    time.sleep(60)